# GENETIC RELATIONSHIP MATRIX: STEP 02

This script prunes array data in the African, American, and European ancestry groups for both GRM calculation and principal component analysis. Variants are first filtered based on a minor allele frequency of 0.01%, a genotyping missingness rate of less than 0.01%, a Hardy-Weinberg equilibrium p-value threshold of 1x10⁻⁶, and must be biallelic and located on autosomal chromosomes. Linkage disequilibrium (LD) pruning is performed within a 200 kb window, removing variants with an r² > 0.1. The output of this script is a file containing variants that pass these filters. We expect the African and American cohorts to have the highest number of variants passing these filters, with the European cohort having the lowest number. The variants that pass these filters will then undergo a second pruning step before being used for GRM calculation and principal component analysis.

In [ ]:
# Load packages 
from datetime import datetime
import pandas as pd 
import os

In [ ]:
# Get the environment variable 'WORKSPACE_BUCKET' which stores your workspace's GCS bucket name
my_bucket = os.getenv("WORKSPACE_BUCKET")
# Workspace bucket 
my_bucket

# COMMAND 

In [ ]:
%%writefile grm_pruning.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # Check if the cohort file for the current population exists
    if [ ! -f "${cohort}/${pop}-like_cohort_qc.txt" ]; then
        echo "Cohort file for ${pop} not found!"
        exit 1
    fi
    
    # Copy new plink module into the plink2 
    cp "${plink2}" plink2

    # Grant permision to run plink2 in environment
    chmod +x plink2
    
    # Check number of samples in each cohort 
    echo "Number of samples in ${pop}:"
    wc -l "${cohort}/${pop}-like_cohort_qc.txt"
    
    # PLINK2 command to perform quality control and LD pruning on SNV variants for the current population

    # - Uses a binary PLINK file as input.
    # - Filters out variants with low minor allele frequency (MAF < 0.01).
    # - Removes variants with high missingness (>1% missing genotypes).
    # - Excludes variants failing Hardy-Weinberg equilibrium (p < 1^-6).
    # - Restricts analysis to autosomal chromosomes (1-22).
    # - Retains only individuals specified in the cohort file (defined by a population file).
    # - Performs linkage disequilibrium (LD) pruning within a 50 step window, removing variants with r² > 0.05
    # - Saves the output files to the specified output directory.
    # - Restricts to single-nucleotide variants (SNVs) and biallelic variants only.
    # - Uses multithreading (4 threads) for faster execution.
    
    # Run PLINK2 for quality control on SNV variants for the current population
    ./plink2 --bfile "${genos}/arrays" \
        --keep "${cohort}/${pop}-like_cohort_qc.txt" \
        --extract "${inclusion_snvs}" \
        --threads 4 \
        --maf 0.01 \
        --geno 0.01 \
        --hwe 1e-6 0 \
        --chr 1-22 \
        --max-alleles 2 \
        --snps-only just-acgt \
        --indep-pairwise 50 5 0.05 \
        --out "${out_path}/${pop}-like_grm_pruned"

done

In [ ]:
# Copy the local script 'grm_pruning.sh' to the 'data/scripts' folder in your workspace GCS bucket
!gsutil cp grm_pruning.sh {my_bucket}/data/scripts/

In [ ]:
# Display the last 20 lines of the 'grm_pruning.sh' script stored in your GCS bucket
!gsutil cat {my_bucket}/data/scripts/grm_pruning.sh 

# SUMBIT COMMAND 

In [ ]:
%%bash --out array_variant_qc 

#############################################################################
# DSUB: SUBMIT SCRIPT FOR VARIANT PRUNING
#############################################################################

# Source the dsub setup script, which likely contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/microarray/plink"

#############################################################################
# SUBMIT DSUB SCRIPT JOB FOR VARIANT PRUNING
#############################################################################

# Use a prebuilt container with PLINK2
# Set total disk space required 
# Define logging directory for dsub execution logs
# Input PLINK files (BED, BIM, FAM) from Google Cloud Storage (GCS)
# Provide cohort sample lists (recursively input from GCS)
# Define output directory for variant QC results (recursively stores outputs)
# Set environment variables for the script
# Specify the script to execute within the container


aou_dsub \
  --disk-size 300 \
  --min-ram 4 \
  --min-cores 4 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/ancestry_sd3_wgs_qc/" \
  --input inclusion_snvs="${WORKSPACE_BUCKET}/data/saige_step00/pruning_files/non_ambiguous_biallelic_snvs.txt" \
  --output-recursive out_path="${WORKSPACE_BUCKET}/data/saige_step00/pruned_for_grm/" \
  --env genos="${mnt_path}" \
  --env cohort="${WORKSPACE_BUCKET}/data/wgs_qc/ancestry_sd3_wgs_qc" \
  --env out_path="${WORKSPACE_BUCKET}/data/saige_step00/pruned_for_grm" \
  --script "${WORKSPACE_BUCKET}/data/scripts/grm_pruning.sh"
  
  
#############################################################################
# END OF SCRIPT 
#############################################################################